In [2]:
#Importing Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import time
import numpy as np

In [3]:
# Device config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, 1)
        self.conv2 = nn.Conv2d(16, 32, 3, 1)
        self.fc1 = nn.Linear(32 * 5 * 5, 64)
        self.fc2 = nn.Linear(64, 10)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(-1, 32 * 5 * 5)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Data transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [5]:
# Load MNIST data
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1000, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 493kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.66MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.31MB/s]


In [33]:
# Instantiate model, loss, optimizer
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
# Train the model
def train(model, train_loader, optimizer, criterion, epochs=5):
    model.train()
    for epoch in range(epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()


# Evaluate overall accuracy + accuracy on class 7
def evaluate(model, test_loader):
    model.eval()
    correct, total = 0, 0
    correct_7, total_7 = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            mask_7 = (labels == 7)
            correct_7 += (predicted[mask_7] == labels[mask_7]).sum().item()
            total_7 += mask_7.sum().item()

    overall_acc = 100 * correct / total
    class7_acc = 100 * correct_7 / total_7 if total_7 > 0 else 0

    return overall_acc, class7_acc


In [34]:
print("Training model...")
train(model, train_loader, optimizer, criterion, epochs=5)

Training model...


In [19]:
before_overall_acc, before_class7_acc = evaluate(model, test_loader)
print(f"Before Unlearning - Overall Accuracy: {before_overall_acc:.2f}%, Class 7 Accuracy: {before_class7_acc:.2f}%")

Before Unlearning - Overall Accuracy: 98.67%, Class 7 Accuracy: 96.98%


## Unlearning via Gradient Ascent

In [21]:
# Approximate unlearning via gradient ascent
def unlearn_class7(model, train_dataset, unlearning_rate=0.001):
    model.train()
    indices_7 = [i for i, (img, label) in enumerate(train_dataset) if label == 7]
    loader_7 = DataLoader(Subset(train_dataset, indices_7), batch_size=128, shuffle=False)
    start_time = time.time()

    for images, labels in loader_7:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        with torch.no_grad():
            for param in model.parameters():
                param.add_(unlearning_rate * param.grad)  # Gradient Ascent step
        model.zero_grad()

    unlearning_time = time.time() - start_time
    return unlearning_time

In [35]:
print("Unlearning class 7...")
unlearning_time = 0
epochs = 3
for epoch in range(epochs):
  unlearning_time += unlearn_class7(model, train_dataset, unlearning_rate=0.0005)

Unlearning class 7...


In [36]:
after_overall_acc, after_class7_acc = evaluate(model, test_loader)
print(f"After Unlearning - Overall Accuracy: {after_overall_acc:.2f}%, Class 7 Accuracy: {after_class7_acc:.2f}%")
print(f"Unlearning Time: {unlearning_time:.2f} seconds")

After Unlearning - Overall Accuracy: 98.16%, Class 7 Accuracy: 93.29%
Unlearning Time: 3.63 seconds


In [37]:
print("Further Unlearning class 7...")
unlearning_time = 0
epochs = 7
for epoch in range(epochs):
  unlearning_time += unlearn_class7(model, train_dataset, unlearning_rate=0.0005)

after_overall_acc, after_class7_acc = evaluate(model, test_loader)
print(f"After Unlearning - Overall Accuracy: {after_overall_acc:.2f}%, Class 7 Accuracy: {after_class7_acc:.2f}%")
print(f"Unlearning Time: {unlearning_time:.2f} seconds")

Further Unlearning class 7...
After Unlearning - Overall Accuracy: 9.80%, Class 7 Accuracy: 0.00%
Unlearning Time: 8.52 seconds


##Catastrophic Collapse